# Indicator Similarity Playbook

Driver notebook for the whole toolkit. Pick a pair of `.pine` scripts,
then run the cells — each one produces its chart inline *and* saves a JPG.

**Run order:** top to bottom. If the dropdowns don't render in your
viewer, edit `FILE_A` / `FILE_B` in the config cell instead.

| step | cell | output |
|---|---|---|
| pair verdict | cell 3 | centered cosine vs thresholds, both models |
| corpus map | cell 4 | `corpus_{heatmap,map}_{0.6b,4b}.jpg` |
| merge sweep | cell 5 | `merge_sweep_{concat,interleave}_*.jpg` (~40 embeds) |
| token-mutation drift | cell 6 | `word_sweep_{fourier,trajectory}_*.jpg` (~20 embeds) |

Requires `ollama serve` + `qwen3-embedding:0.6b` pulled. Full writeup:
`SEMANTIC_SIMILARITY.md`.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path.cwd()))
import embed_indicators as ei
import corpus_map
import merge_experiment
import word_sweep_fourier

SCRIPTS = sorted(f.name for f in ei.INDICATOR_DIR.glob('*.pine'))
print(f'{len(SCRIPTS)} .pine files in {ei.INDICATOR_DIR.name}/')

In [ ]:
# ---- CONFIG: pick the pair -----------------------------------------------
# Defaults if the dropdowns can't render in this viewer.
FILE_A = '08_rsi_divergence.pine'   # RSI-based
FILE_B = 'macd_ema.pine'           # EMA-based

HAS_WIDGETS = False
try:
    import ipywidgets as widgets
    from IPython.display import display
    dd_a = widgets.Dropdown(options=SCRIPTS, value=FILE_A, description='A:',
                            layout=widgets.Layout(width='420px'),
                            style={'description_width': '30px'})
    dd_b = widgets.Dropdown(options=SCRIPTS, value=FILE_B, description='B:',
                            layout=widgets.Layout(width='420px'),
                            style={'description_width': '30px'})

    def _sync(*_):
        global FILE_A, FILE_B
        FILE_A, FILE_B = dd_a.value, dd_b.value

    dd_a.observe(_sync, 'value')
    dd_b.observe(_sync, 'value')
    HAS_WIDGETS = True
    display(widgets.VBox([
        dd_a, dd_b,
        widgets.HTML('<i>the verdict cell updates live — re-run the sweep '
                     'cells manually after changing the selection</i>'),
    ]))
except Exception as e:
    print(f'ipywidgets unavailable ({e}) — edit FILE_A/FILE_B above instead')
    print('\n'.join(SCRIPTS))

In [ ]:
def pair_verdict(a, b):
    """Centered cosine of the pair in every available model DB."""
    print(f'A: {a}\nB: {b}\n')
    print(f"{'model':<6} {'cosine':>8}   verdict")
    print('-' * 60)
    for label, path in corpus_map.MODELS.items():
        if not path.exists():
            print(f'{label:<6}   n/a     ({path.name} missing)')
            continue
        db = json.loads(path.read_text())
        inds = db['indicators']
        if a not in inds or b not in inds:
            print(f'{label:<6}   n/a     (pair not embedded)')
            continue
        raw = {n: np.asarray(e['vector']) for n, e in inds.items()}
        if 'reference_mean' in db:
            mu = np.asarray(db['reference_mean'])
        else:
            mu = np.stack(list(raw.values())).mean(axis=0)
        s = ei.cosine(ei.center(raw[a], mu), ei.center(raw[b], mu))
        tag = ('REMIX' if s >= ei.REMIX_THRESHOLD
               else 'related' if s >= ei.RELATED_THRESHOLD else 'distinct')
        print(f'{label:<6} {s:>8.4f}   {tag}')

if HAS_WIDGETS:
    display(widgets.interactive_output(pair_verdict, {'a': dd_a, 'b': dd_b}))
else:
    pair_verdict(FILE_A, FILE_B)

In [ ]:
# All-vs-all: clustered similarity heatmap + labelled PCA map, both models.
# No embedding needed — uses the stored vectors. Labels show `// @title:`.
corpus_map.run('0.6b')
corpus_map.run('4b')

In [ ]:
# Merge sweep: splice A and B at 0->100%, embed each hybrid, plot the
# centered cosine vs each parent against the REMIX/RELATED thresholds.
# ~21 live embeds per mode — a couple of minutes on 0.6b.
merge_experiment.run(FILE_A, FILE_B, mode='concat')
merge_experiment.run(FILE_A, FILE_B, mode='interleave')

In [ ]:
# Token-mutation drift: replace A's tokens with B's one by one, embed each
# mutant -> contribution erosion heatmap + 2D PCA trajectory map.
# ~22 live embeds; results cached per pair in word_sweep_cache_<a>_vs_<b>.npz
word_sweep_fourier.sweep_pine(FILE_A, FILE_B)

In [ ]:
# Every JPG this toolkit has produced, newest first.
from datetime import datetime
for p in sorted(Path('.').glob('*.jpg'), key=lambda p: -p.stat().st_mtime):
    print(f'{p.name:<66} {datetime.fromtimestamp(p.stat().st_mtime):%H:%M:%S}'
          f'  {p.stat().st_size // 1024} KB')